

### Overview

We will build a pipeline in Google Colab that performs the following steps:

1. **Environment Setup:** We will mount Google Drive to access the `DFT-Corpus-25` dataset and load Hugging Face token from Colab Secrets.
2. **Model Selection:** We will use **Mistral-7B-Instruct-v0.3**. It is a powerful, free, open-weights LLM that fits on the free Colab T4 GPU when loaded with 4-bit quantization. It is also excellent at following strict formatting instructions like your CSV schema.
3. **Document Loading & Chunking:** Scientific papers are too long for a single LLM prompt. We will read the PDFs and split them into manageable chunks (pages or token blocks).
4. **Extraction Pipeline:** We will loop through the chunks, injecting the text into your meticulously crafted prompt, and ask the LLM to generate the CSV blocks.
5. **Parsing & Saving:** We will use Python string manipulation to separate the LLM's output into `nodes.csv` and `relationships.csv` and save them for your downstream analysis.

---

### Implementation Instructions

Create a new Google Colab notebook, make sure your runtime is set to **T4 GPU** (Runtime > Change runtime type > Hardware accelerator > T4 GPU), and paste the following blocks of code into separate cells.

####  Install Dependencies

First, we need to install the required libraries for reading PDFs, running the LLM, and applying 4-bit quantization to fit the model into the GPU memory.

In [ ]:
# Install required libraries
!pip install -q transformers accelerate bitsandbytes pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.5 MB/s eta 0:00:00


####  Setup Drive and Secrets

This cell mounts your Google Drive and securely retrieves your Hugging Face token. Make sure your secret is named exactly `HF_TOKEN` in the Colab Secrets tab (the key icon on the left sidebar) and that "Notebook access" is toggled on.

In [ ]:
import os
from google.colab import drive
from google.colab import userdata

# Mount Google Drive
drive.mount('/content/drive')

# Retrieve Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Set the path to your dataset
DATASET_PATH = "/content/drive/MyDrive/Dissertation/Datasets/DFT-Corpus-25"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


####  Initialize the LLM

Here, we load the Mistral model. We use `bitsandbytes` to load it in 4-bit precision, which drastically reduces the memory footprint.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# We use Mistral-7B-Instruct as it is highly capable and free
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

print("Loading model in 4-bit quantization...")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    token=hf_token,
    torch_dtype=torch.float16 # Keep this for other parts of the model if not handled by bnb_config
)

# Create the text generation pipeline
# We set a high max_new_tokens to ensure the LLM doesn't cut off the CSV generation
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1500,
    temperature=0.1, # Low temperature for highly deterministic extraction
    return_full_text=False
)
print("Pipeline ready!")

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Pipeline ready!


####  Document Loading and Chunking

This code reads all the PDFs in your target folder and chunks them by page to feed them to the LLM sequentially.

In [ ]:
import glob
from pypdf import PdfReader

def get_pdf_texts(folder_path):
    """Reads all PDFs in a folder and returns a list of dictionaries with text chunks."""
    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    documents = []

    print(f"Found {len(pdf_files)} PDF files.")

    for file_path in pdf_files:
        try:
            reader = PdfReader(file_path)
            paper_title = os.path.basename(file_path)

            # Group text by pages (you can also chunk by character/token length if preferred)
            for page_num, page in enumerate(reader.pages):
                text = page.extract_text()
                if text and len(text.strip()) > 100: # Ignore mostly empty pages
                    documents.append({
                        "paper": paper_title,
                        "page": page_num + 1,
                        "text": text
                    })
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    return documents

# Load the documents
docs = get_pdf_texts(DATASET_PATH)
print(f"Total chunks to process: {len(docs)}")

Found 25 PDF files.


Total chunks to process: 567


####  The Extraction Engine

This is the core loop. It injects your prompt instructions and the text chunk into the model, then extracts the generated CSV blocks.

In [ ]:
import pandas as pd
import re

# Your expert extraction instructions
SYSTEM_PROMPT = """You are an expert scientific data extractor for computational chemistry. You never miss even the smallest details. From the document you are analyzing, extract all entities and relationships according to the schema below.
SCHEMA:
Nodes: Paper, Functional, DispersionCorrection, BenchmarkSet, ValidationResult, Metric
Relationships: CITES, DEVELOPS, REFINES, VALIDATES, APPLIES_CORRECTION, TESTS_ON, REPORTS_RESULT, RESULT_FOR_METHOD, RESULT_FOR_BENCHMARK, RESULT_USING_METRIC

CRITICAL INSTRUCTIONS:
- Functionals: For each Functional, must include properties: name, rung, and class (e.g., 'PBE', 2, 'GGA').
- Corrections: Split corrected functionals (e.g., "PBE-D3") into two entities: Functional {name: 'PBE'} and DispersionCorrection {name: 'D3'}.
- Validation Results: Capture ValidationResult nodes with all their connections.
- Evolution: If the paper mentions one method improves another (e.g., "D4 builds on D3"), add a REFINES relationship.

OUTPUT FORMAT: Provide the output ONLY as two separate CSV-formatted blocks.
1. nodes.csv block: Columns: node_id, label, rung, class, family, unit, value (Use the name or title as the unique node_id. Fill other properties only where applicable.)
2. relationships.csv block: Columns: source_id, target_id, relationship_type (Use the node_ids from the nodes block for source_id and target_id.)

DO NOT output any conversational text. ONLY output the two CSV blocks.
"""

all_nodes = []
all_relationships = []

# For demonstration, we'll process the first 3 chunks to ensure it works.
# Remove "[:3]" to process all chunks once you verify the output.
for i, doc in enumerate(docs[:3]):
    print(f"Processing chunk {i+1} from {doc['paper']}...")

    user_message = f"DOCUMENT TEXT:\n{doc['text']}\n\nEXTRACT THE CSV BLOCKS NOW."

    # Format messages for Mistral
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_message}
    ]

    # Generate output
    output = pipe(messages)
    generated_text = output[0]['generated_text']

    # --- Parsing Logic ---
    # We use basic string manipulation to find the two blocks based on headers
    try:
        # Split by the known headers
        if "node_id,label,rung,class,family,unit,value" in generated_text:
            node_part = generated_text.split("node_id,label,rung,class,family,unit,value")[1].split("source_id,target_id,relationship_type")[0]
            rel_part = generated_text.split("source_id,target_id,relationship_type")[1]

            # Clean up the parsed strings
            node_lines = [line.strip() for line in node_part.split('\n') if line.strip() and not line.startswith('```')]
            rel_lines = [line.strip() for line in rel_part.split('\n') if line.strip() and not line.startswith('```')]

            all_nodes.extend(node_lines)
            all_relationships.extend(rel_lines)
    except Exception as e:
        print(f"Could not parse chunk {i+1}: {e}")

print("Extraction complete!")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing chunk 1 from Density-fnnctional exchange-energy approximation with correct asymptotic behavior.pdf...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing chunk 2 from Density-fnnctional exchange-energy approximation with correct asymptotic behavior.pdf...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing chunk 3 from Density-fnnctional exchange-energy approximation with correct asymptotic behavior.pdf...
Extraction complete!


####  Compiling and Saving the Results

Finally, we take all the collected CSV lines, convert them into pandas DataFrames to clean up duplicates, and save them as actual `.csv` files.

In [ ]:
# Create DataFrames
node_header = ["node_id", "label", "rung", "class", "family", "unit", "value"]
rel_header = ["source_id", "target_id", "relationship_type"]

# Parse the accumulated lines into lists of lists
parsed_nodes = [line.split(",") for line in all_nodes if len(line.split(",")) == 7]
parsed_rels = [line.split(",") for line in all_relationships if len(line.split(",")) == 3]

df_nodes = pd.DataFrame(parsed_nodes, columns=node_header)
df_rels = pd.DataFrame(parsed_rels, columns=rel_header)

# Drop duplicates (since the same entity might be extracted across multiple chunks)
df_nodes.drop_duplicates(subset=["node_id"], inplace=True)
df_rels.drop_duplicates(inplace=True)

# Save to your Drive
nodes_path = "/content/drive/MyDrive/Dissertation/Datasets/extracted_nodes.csv"
rels_path = "/content/drive/MyDrive/Dissertation/Datasets/extracted_relationships.csv"

df_nodes.to_csv(nodes_path, index=False)
df_rels.to_csv(rels_path, index=False)

# Display a preview
display(df_nodes.head())
display(df_rels.head())

,node_id,label,rung,class,family,unit,value
0,"""Becke_1988""","""Paper""","""1""","""Theory""","""DFT""","""""",""""""
1,"""Exchange""",Functional,"""2""","""GGA""","""Exchange Functional""",,"""PBE"""
2,"""Density""",Functional,"""2""","""GGA""","""Exchange Functional""",,"""pw"""
3,"""He""",BenchmarkSet,"""""",,"""Noble Gases""",,""""""
4,"""Ne""",BenchmarkSet,"""""",,"""Noble Gases""",,""""""


,source_id,target_id,relationship_type
0,"""Becke_1988""","""LDA""","""CITES"""
1,"""Becke_1988""","""SE""","""DEVELOPS"""
2,"""Becke_1988""","""LGC""","""DEVELOPS"""
3,"""Becke_1988""","""x""","""APPLIES_CORRECTION"""
4,"""Becke_1988""","""p""","""APPLIES_CORRECTION"""
